[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/templates/55_weight_decay.ipynb)

# 🟢 Easy: Weight Decay (L2 in gradient)

**Weight decay** を実装する。L2 正則化項を optimizer step の **前** に gradient に 足す（SGD/L2 規約）。AdamW は別の (decoupled) formulation を使う (#56 参照)。

> 💡 **どこで使う**：重みを 0 に引き寄せる正則化。SGD では gradient に `λp` を足す形（L2 ペナルティ）。

<details>
<summary>📖 もっと詳しく</summary>

loss に `(λ/2)·||p||²` を足すのと数学的に等価、ただし optimizer step で直接やる方が計算グラフを増やさない。

Adam に L2 をこの形で適用すると `√v_hat` で割られて weight ごとに実効 decay が変わる → AdamW (#56) の登場。

</details>

### L2-in-gradient 形式（この問題）
$$g \leftarrow g + \lambda \cdot p$$

loss に `(λ/2)·‖p‖²` を足して autograd に gradient へ `λ·p` を足させるのと等価。optimizer step で直接やればグラフ node が増えない分速い。

### vs. AdamW の decoupled 形式
$$p \leftarrow p \cdot (1 - \text{lr} \cdot \lambda)$$
gradient 経由じゃなく **param** に直接適用。Loshchilov & Hutter (2019) が adaptive optimizer ではこっちが効くと示した。


### Signature
```python
def apply_weight_decay(params, weight_decay):
    # params: a single tensor OR iterable of tensors
    # weight_decay: float λ
    # In-place: adds weight_decay * p to each p.grad
    # Returns: None
```

### Rules
- **In-place**: `p.grad` を直接 modify、戻り値なし
- Single tensor / list / iterable いずれも受け付ける
- `p.grad is None` の param は skip
- `weight_decay == 0` なら何もしない
- `@torch.no_grad()` で wrap — `.grad` を modify するだけ、autograd 不要

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_weight_decay(params, weight_decay):
    pass  # no_grad 下で: p.grad がある各 p に対し p.grad.add_(p, alpha=weight_decay)

In [ ]:
import torch
p = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
p.grad = torch.ones(3)
apply_weight_decay(p, weight_decay=0.1)
print('grad after WD (was ones, +0.1*p):', p.grad.tolist())

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("weight_decay")

---

**詰まったら？** 解答を見る：

[![Open Solution in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/55_weight_decay_solution.ipynb) または [GitHub で開く](https://github.com/alextfkd/TorchCode/blob/master/solutions/55_weight_decay_solution.ipynb)